In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# hide
def lpc(x: np.ndarray, K: int) -> np.ndarray:
    """The coefficients h_1, ..., h_K that best predict each x[n] from x[n-1], ..., x[n-K]."""
    N = len(x)
    X = np.stack([x[K - k:N - k] for k in range(1, K + 1)], axis=1)   # row n: x[n-1], ..., x[n-K]
    h, *_ = np.linalg.lstsq(X, x[K:], rcond=None)                     # minimize the squared error
    return h

In [ ]:
def pulse_train(f0: float, duration: float, f_s: int) -> np.ndarray:
    """A train of unit impulses, f0 of them per second: a crude glottal pulse train."""
    e = np.zeros(int(duration * f_s))
    e[::round(f_s / f0)] = 1.0
    return e


# An LPC filter for each vowel
paths = {"A": "../assets/vowel-a.wav", "E": "../assets/vowel-e.wav", "I": "../assets/vowel-i.wav",
         "O": "../assets/vowel-o.wav", "U": "../assets/vowel-u.wav"}
filters = {}
for name, path in paths.items():
    vowel = pq.Audio.from_file(path)
    x = vowel.segment(offset=1.0, duration=0.5).samples[:, 0].astype(np.float64)
    filters[name] = np.concatenate([[1.0], -lpc(x, K=40)])
f_s = vowel.sample_rate


def sing(pitch: int, vowel: str, duration: float, f_s: int) -> np.ndarray:
    """One note: a pulse train at the pitch, faded in and out, through the vowel's filter."""
    e = pulse_train(pq.helper.pitch_to_frequency(pitch), duration, f_s)
    n = np.arange(len(e))
    envelope = np.minimum(1.0, np.minimum(n / (0.02 * f_s), (len(e) - n) / (0.1 * f_s)))
    return lfilter([1.0], filters[vowel], envelope * e)


# Sing the first phrase of Twinkle, Twinkle, Little Star, changing vowels as we go
melody = [48, 48, 55, 55, 57, 57, 55]
vowels = "AAEEIOU"
durations = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0]
y = np.concatenate([sing(pitch, vowel, duration, f_s)
                    for pitch, vowel, duration in zip(melody, vowels, durations)])
pq.play(pq.Audio(y, f_s), normalize=True)